# Vector-space models — runnable notebook
One focused concept, **5 runnable & visualizable examples** — each cell computes, plots, and asserts. Run-all safe.

In [ ]:
import numpy as np, matplotlib.pyplot as plt, math, itertools
np.random.seed(0)
def cos(a,b):
    a=np.asarray(a,float); b=np.asarray(b,float)
    return float(np.dot(a,b)/(np.linalg.norm(a)*np.linalg.norm(b)))
print('setup ok')

## TF-IDF vectors rank documents by cosine alignment
The examples build the document-term matrix, apply IDF weights, compare dot product with cosine, rank documents, and show how common terms distort geometry without IDF.

In [ ]:
# 1) Build the document-term matrix
vocab=['neural','search','graph','retrieval']
corpus=[['neural','search','search'],['graph','search'],['neural','retrieval','models'],['graph','retrieval','search','search']]
X=np.array([[d.count(t) for t in vocab] for d in corpus],float)
plt.figure(figsize=(6,3)); plt.imshow(X,cmap='Blues'); plt.xticks(range(4),vocab); plt.yticks(range(4),['d1','d2','d3','d4']); plt.colorbar(label='count'); plt.title('document-term matrix')
assert X.shape==(4,4) and X[0,1]==2

In [ ]:
# 2) Apply smoothed IDF weights
N=4; df=(X>0).sum(axis=0); idf=np.log((N+1)/(df+1))+1; V=X*idf
plt.figure(figsize=(6,3)); plt.bar(vocab,idf); plt.ylabel('idf'); plt.title('search is down-weighted because df=3')
assert np.allclose(idf,[1.51082562,1.22314355,1.51082562,1.51082562])

In [ ]:
# 3) Cosine removes the advantage of vector length
q=np.array([1,1,0,0])*idf
dot=[float(np.dot(q,v)) for v in V]; cosine=[cos(q,v) for v in V]
plt.figure(figsize=(6,3)); plt.plot(['d1','d2','d3','d4'],dot,'-o',label='dot'); plt.plot(['d1','d2','d3','d4'],cosine,'-s',label='cosine'); plt.legend(); plt.title('dot rewards size; cosine rewards direction')
assert cosine[0]>cosine[2]>cosine[3]>cosine[1]

In [ ]:
# 4) Rank documents by TF-IDF cosine
order=np.argsort(cosine)[::-1]
plt.figure(figsize=(6,3)); plt.bar(['d1','d2','d3','d4'],cosine); plt.ylabel('cosine'); plt.title('TF-IDF cosine ranking')
assert list(order)==[0,2,3,1] and abs(cosine[0]-0.9437583081)<1e-9

In [ ]:
# 5) Without IDF, common search repetitions pull the geometry
raw_q=np.array([1,1,0,0],float); raw_cos=[cos(raw_q,x) for x in X]
plt.figure(figsize=(6,3)); plt.bar(np.arange(4)-.15,raw_cos,width=.3,label='raw'); plt.bar(np.arange(4)+.15,cosine,width=.3,label='tf-idf'); plt.xticks(range(4),['d1','d2','d3','d4']); plt.legend(); plt.title('weighting changes the sparse geometry')
assert raw_cos[1]>cosine[1] and cosine[2]>raw_cos[2]